In [1]:
import csv
import json
import os
from collections import defaultdict

In [2]:
# Dossiers
DOSSIER = "Adjudication"
FICHIERS_CONSOLIDES = [
    "adjudication_CARRAUD_petite-Jeanne_Kraken-base_spaCy3.7.2-lg 1_consolide.csv",
    "adjudication_CARRAUD_petite-Jeanne_REF_spaCy3.7.2-lg_consolide.csv",
    "adjudication_CARRAUD_petite-Jeanne_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv",
    "adjudication_DAUDET_petit-chose_Kraken-base_spaCy3.7.2-lg_consolide.csv",
    "adjudication_DAUDET_petit-chose_REF_spaCy3.7.2-lg_consolide.csv",
    "adjudication_DAUDET_petit-chose_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv"
]

In [3]:
# Fichier de sortie JSON
CHEMIN_SORTIE = "Corpus_Entrainement_Enrichi_avec_Quaero_fusioné.json"

In [4]:
TAG_ID = {"PER": 1, "LOC": 2, "ORG": 3, "MISC": 4}

corpus = []
annotation_global_id = 1

for fichier in FICHIERS_CONSOLIDES:
    chemin = os.path.join(DOSSIER, fichier)
    base_name = fichier.replace("_consolide.csv", ".csv")
    chemin_complet_base = os.path.join(DOSSIER, base_name)

    with open(chemin, newline='', encoding='utf-8') as f_maj, open(chemin_complet_base, newline='', encoding='utf-8') as f_base:
        reader_maj = csv.DictReader(f_maj, delimiter=';', quotechar='"')
        reader_base = csv.DictReader(f_base, delimiter=';', quotechar='"')

        tokens = []
        annotations = []
        position = 0

        for ligne_maj, ligne_base in zip(reader_maj, reader_base):
            token = ligne_maj["Token"]
            label_maj = ligne_maj["Label_Maj"].strip()
            label_spacy = ligne_base["Label"].strip()
            quaero_maj = ligne_maj.get("Quaero_Maj", "").strip()

            # Récupère les annotations humaines (exclut NA)
            human_anns = [v.strip() for k, v in ligne_base.items() if "Correction" in k and v.strip() not in ["", "NA"]]

            # Offsets
            start = position
            end = position + len(token)

            if label_maj:
                # Fusion LOC + Quaero
                if label_maj == "LOC" and quaero_maj:
                    label_final = f"{label_maj}_{quaero_maj}"  # ex: LOC_loc.adm.town
                else:
                    label_final = label_maj

                annotation = {
                    "id": annotation_global_id,
                    "start_offset": start,
                    "end_offset": end,
                    "tag": {
                        "id": TAG_ID.get(label_maj, 99),  # on garde l'ID de base (utile pour compat)
                        "name": label_final
                    },
                    "tag_name": label_final,
                    "model_annotations": [label_spacy] if label_spacy else [],
                    "human_annotations": human_anns
                }

                annotations.append(annotation)
                annotation_global_id += 1

            tokens.append(token)
            position = end + 1  # +1 pour l'espace simulé

        content = " ".join(tokens)
        doc_id = fichier.replace(".csv", "")

        corpus.append({
            "id": doc_id,
            "content": content,
            "metadata": {},
            "annotations": annotations
        })

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as fout:
    json.dump(corpus, fout, indent=2, ensure_ascii=False)

print(f"JSON généré : {CHEMIN_SORTIE}")

JSON généré : Corpus_Entrainement_Enrichi_avec_Quaero_fusioné.json


In [2]:
import csv
import json
import os
from collections import defaultdict
from math import floor

# Dossiers
DOSSIER = "Adjudication"
DOSSIER_SORTIE = "quaero_enrichi"
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

FICHIERS_CONSOLIDES = [
    "adjudication_CARRAUD_petite-Jeanne_Kraken-base_spaCy3.7.2-lg 1_consolide.csv",
    "adjudication_CARRAUD_petite-Jeanne_REF_spaCy3.7.2-lg_consolide.csv",
    "adjudication_CARRAUD_petite-Jeanne_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv",
    "adjudication_DAUDET_petit-chose_Kraken-base_spaCy3.7.2-lg_consolide.csv",
    "adjudication_DAUDET_petit-chose_REF_spaCy3.7.2-lg_consolide.csv",
    "adjudication_DAUDET_petit-chose_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv"
]

TAG_ID = {"PER": 1, "LOC": 2, "ORG": 3, "MISC": 4}
OCR_TYPES = ["REF", "Kraken", "Tesseract"]
PERCENTAGES = [25, 50, 75]

corpus_total = []
corpus_by_ocr = {ocr: [] for ocr in OCR_TYPES}
annotation_global_id = 1

for fichier in FICHIERS_CONSOLIDES:
    chemin = os.path.join(DOSSIER, fichier)
    base_name = fichier.replace("_consolide.csv", ".csv")
    chemin_base = os.path.join(DOSSIER, base_name)

    with open(chemin, newline='', encoding='utf-8') as f_maj, open(chemin_base, newline='', encoding='utf-8') as f_base:
        reader_maj = csv.DictReader(f_maj, delimiter=';', quotechar='"')
        reader_base = csv.DictReader(f_base, delimiter=';', quotechar='"')

        tokens, annotations, position = [], [], 0
        for ligne_maj, ligne_base in zip(reader_maj, reader_base):
            token = ligne_maj["Token"]
            label_maj = ligne_maj["Label_Maj"].strip()
            label_spacy = ligne_base["Label"].strip()
            quaero_maj = ligne_maj.get("Quaero_Maj", "").strip()
            human_anns = [v.strip() for k, v in ligne_base.items() if "Correction" in k and v.strip() not in ["", "NA"]]
            start, end = position, position + len(token)

            if label_maj:
                label_final = f"{label_maj}_{quaero_maj}" if label_maj == "LOC" and quaero_maj else label_maj
                annotation = {
                    "id": annotation_global_id,
                    "start_offset": start,
                    "end_offset": end,
                    "tag": {"id": TAG_ID.get(label_maj, 99), "name": label_final},
                    "tag_name": label_final,
                    "model_annotations": [label_spacy] if label_spacy else [],
                    "human_annotations": human_anns
                }
                annotations.append(annotation)
                annotation_global_id += 1

            tokens.append(token)
            position = end + 1

        doc_id = fichier.replace(".csv", "")
        content = " ".join(tokens)
        entry = {"id": doc_id, "content": content, "metadata": {}, "annotations": annotations}

        corpus_total.append(entry)

        for ocr in OCR_TYPES:
            if ocr.lower() in fichier.lower():
                corpus_by_ocr[ocr].append(entry)

# Fonction d’écriture JSON
def ecrire_json(nom, data):
    chemin = os.path.join(DOSSIER_SORTIE, nom)
    with open(chemin, "w", encoding="utf-8") as fout:
        json.dump(data, fout, indent=2, ensure_ascii=False)
    print(f"✔️ JSON généré : {chemin}")

# Fichier principal (100%)
ecrire_json("Corpus_Entrainement_Enrichi_avec_Quaero_fusioné.json", corpus_total)

# Fichiers OCR (REF, Kraken, Tesseract)
for ocr, data in corpus_by_ocr.items():
    ecrire_json(f"Corpus_Entrainement_Enrichi_{ocr}.json", data)

# Fichiers avec pourcentages (25%, 50%, 75%) du total
for pct in PERCENTAGES:
    taille = floor(len(corpus_total) * pct / 100)
    sous_ensemble = []

    for fichier in FICHIERS_CONSOLIDES:
        doc = [d for d in corpus_total if d["id"] == fichier.replace(".csv", "")]
        if doc:
            sous_ensemble.extend(doc[:1])  # on prend au moins 1 entrée par fichier

    while len(sous_ensemble) < taille:
        for doc in corpus_total:
            if doc not in sous_ensemble:
                sous_ensemble.append(doc)
            if len(sous_ensemble) == taille:
                break

    ecrire_json(f"Corpus_Entrainement_Enrichi_{pct}p.json", sous_ensemble)

✔️ JSON généré : quaero_enrichi/Corpus_Entrainement_Enrichi_avec_Quaero_fusioné.json
✔️ JSON généré : quaero_enrichi/Corpus_Entrainement_Enrichi_REF.json
✔️ JSON généré : quaero_enrichi/Corpus_Entrainement_Enrichi_Kraken.json
✔️ JSON généré : quaero_enrichi/Corpus_Entrainement_Enrichi_Tesseract.json
✔️ JSON généré : quaero_enrichi/Corpus_Entrainement_Enrichi_25p.json
✔️ JSON généré : quaero_enrichi/Corpus_Entrainement_Enrichi_50p.json
✔️ JSON généré : quaero_enrichi/Corpus_Entrainement_Enrichi_75p.json
